# Estudo de classificação de inadimplência e de tratamento de desbalanceamento

Este projeto tem como objetivo praticar e aprender técnicas de Machine Learning, com foco no balanceamento de dados utilizando SMOTENC, na busca de hiperparâmetros com GridSearchCV e no uso de Pipelines para evitar data leakage.

## Dados

Os dados utilizados são da amostra bancária disponibilizada por Cibele Russo: https://github.com/cibelerusso/Datasets/blob/main/amostra_banco.csv

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('https://raw.githubusercontent.com/cibelerusso/Datasets/main/amostra_banco.csv')
df.head()

In [ ]:
df.describe()
df.shape
df.isnull().sum()

Não há dados faltantes.

In [ ]:
df['Inadimplente'].value_counts()
sns.countplot(data=df, x='Inadimplente')
plt.title('Distribuição da variável Inadimplente')
plt.show()

## Análise exploratória

Vamos comparar as distribuições das variáveis quantitativas entre clientes inadimplentes e não inadimplentes.

In [ ]:
numerical_cols = df.select_dtypes(include='number').columns
vars_quant = [c for c in numerical_cols if c not in ['Inadimplente', 'Unnamed: 0']]

rows = (len(vars_quant) + 1) // 2
fig, axes = plt.subplots(nrows=rows, ncols=2, figsize=(15, 5 * rows))
axes = axes.flatten()

for i, col in enumerate(vars_quant):
    sns.histplot(data=df, x=col, hue='Inadimplente', multiple='stack', kde=True, palette='viridis', ax=axes[i])
    axes[i].set_title(f'Distribuição de {col} por Inadimplência')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Contagem')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

Observa-se que as distribuições de idade e salário apresentam forte sobreposição entre clientes inadimplentes e não inadimplentes. Para o saldo em conta corrente e a dívida de cartão, entretanto, observam-se diferenças entre as distribuições dos dois grupos. As variáveis de poupança e investimento apresentam forte assimetria, com grande concentração de valores próximos de zero e a presença de alguns valores elevados.

## Modelo inicial: Random Forest

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

categorical_cols = df.select_dtypes(exclude='number').columns
df_dummies = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_dummies.drop(['Inadimplente', 'Unnamed: 0', 'Cliente'], axis=1)
y = df_dummies['Inadimplente']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

forest = RandomForestClassifier(n_estimators=100, random_state=0, max_depth=12)
forest.fit(X_train, y_train)
y_pred = forest.predict(X_test)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f'Accuracy: {forest.score(X_test, y_test):.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-score: {f1:.4f}')

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Não Inadimplente', 'Inadimplente'], yticklabels=['Não Inadimplente', 'Inadimplente'])
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de Confusão')
plt.show()

### Análise das métricas

A precisão é baixa, indicando muitos falsos positivos. O recall também é baixo: o modelo identifica apenas uma parcela dos clientes que realmente são inadimplentes. A matriz de confusão ajuda a visualizar esses erros.

In [ ]:
print(f"Número de clientes inadimplentes: {len(df[df['Inadimplente'] == 1])}")
print(f"Número de clientes não inadimplentes: {len(df[df['Inadimplente'] == 0])}")

Os clientes inadimplentes formam uma classe minoritária. Para tentar melhorar a identificação dessa classe, será utilizado o SMOTENC.

## Balanceamento com SMOTENC

In [ ]:
from imblearn.over_sampling import SMOTENC

X_mixed = df.drop(['Inadimplente', 'Unnamed: 0', 'Cliente'], axis=1)
categorical_features = X_mixed.select_dtypes(exclude='number').columns.tolist()

X_train_mixed, X_test_mixed, y_train_mixed, y_test_mixed = train_test_split(X_mixed, y, random_state=42)

sm = SMOTENC(categorical_features=categorical_features, random_state=42)
X_train_resampled, y_train_resampled = sm.fit_resample(X_train_mixed, y_train_mixed)

print('Antes do SMOTENC:')
print(y_train_mixed.value_counts())
print('Depois do SMOTENC:')
print(y_train_resampled.value_counts())

In [ ]:
X_train_resampled_encoded = pd.get_dummies(X_train_resampled, columns=categorical_features, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_mixed, columns=categorical_features, drop_first=True)

X_train_resampled_encoded, X_test_encoded = X_train_resampled_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)
X_test_encoded = X_test_encoded[X_train_resampled_encoded.columns]

forest_resampled = RandomForestClassifier(n_estimators=100, random_state=0, max_depth=12)
forest_resampled.fit(X_train_resampled_encoded, y_train_resampled)

y_pred_resampled = forest_resampled.predict(X_test_encoded)

from sklearn.metrics import accuracy_score
accuracy_resampled = accuracy_score(y_test, y_pred_resampled)
precision_resampled = precision_score(y_test, y_pred_resampled)
recall_resampled = recall_score(y_test, y_pred_resampled)
f1_resampled = f1_score(y_test, y_pred_resampled)

print(f'Accuracy (Resampled): {accuracy_resampled:.4f}')
print(f'Precision (Resampled): {precision_resampled:.4f}')
print(f'Recall (Resampled): {recall_resampled:.4f}')
print(f'F1-score (Resampled): {f1_resampled:.4f}')

In [ ]:
cm_resampled = confusion_matrix(y_test, y_pred_resampled)
plt.figure(figsize=(6, 4))
sns.heatmap(cm_resampled, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Não Inadimplente', 'Inadimplente'], yticklabels=['Não Inadimplente', 'Inadimplente'])
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de Confusão (Modelo com SMOTENC)')
plt.show()

## Otimização de hiperparâmetros com GridSearchCV

O SMOTENC e o pré-processamento serão colocados dentro de um Pipeline, para que o oversampling seja realizado dentro de cada divisão da validação cruzada, evitando data leakage entre treino e validação.

In [ ]:
from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV

preprocessor = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)], remainder='passthrough')

pipeline = Pipeline([
    ('smote', SMOTENC(categorical_features=categorical_features, random_state=42)),
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=0))
])

param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [5, 10, 15, None],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=5, scoring='precision', n_jobs=-1, verbose=1)
grid_search.fit(X_train_mixed, y_train_mixed)

print(f"Melhores parâmetros encontrados: {grid_search.best_params_}")
print(f"Melhor precision na validação cruzada: {grid_search.best_score_:.4f}")

melhor_modelo = grid_search.best_estimator_
y_pred_otimizado = melhor_modelo.predict(X_test_mixed)

In [ ]:
accuracy_otimizado = accuracy_score(y_test, y_pred_otimizado)
precision_otimizado = precision_score(y_test, y_pred_otimizado)
recall_otimizado = recall_score(y_test, y_pred_otimizado)
f1_otimizado = f1_score(y_test, y_pred_otimizado)

print(f'Acurácia (Otimizado): {accuracy_otimizado:.4f}')
print(f'Precision (Otimizado): {precision_otimizado:.4f}')
print(f'Recall (Otimizado): {recall_otimizado:.4f}')
print(f'F1-score (Otimizado): {f1_otimizado:.4f}')

# Conclusões

Neste projeto, o modelo conseguiu identificar alguns clientes inadimplentes, mas apresentou bastante dificuldade para encontrar essa classe corretamente. No modelo inicial, a acurácia ficou relativamente alta, mas o recall para os inadimplentes foi baixo.

Ao aplicar o SMOTENC, o modelo passou a identificar uma parcela maior dos clientes inadimplentes. O recall aumentou de 14,81% para 29,63%, e o F1-score também melhorou. Por outro lado, essa melhora veio acompanhada de uma redução na acurácia e na precisão.

Nesse problema, o recall é especialmente importante, pois indica quantos dos clientes que realmente são inadimplentes foram identificados pelo modelo. Por isso, olhar apenas para a acurácia não seria suficiente.

Como próximos passos, seria interessante testar outros modelos, explorar melhor os hiperparâmetros e experimentar diferentes formas de lidar com o desbalanceamento.